# Desnormalização e Agrupamento Hierárquico de Gêneros

Neste notebook, realizamos o pré-processamento para reorganizar os gêneros literários do dataset. O objetivo é:
1. **Desnormalizar o dataset** (fazer a explosão de gêneros) de forma que cada linha represente uma associação única entre um livro e um gênero.
2. **Realizar um agrupamento hierárquico** de mais de 1.100 subgêneros em **9 Grandes Categorias (Gêneros Principais)**, tratando subgêneros raros como "Outros".
3. **Analisar as métricas gerais** dessas novas categorias de gêneros principais.
4. **Demonstrar as abordagens** para subsidiar a etapa de clusterização futura (Dataset Explodido vs. Dataset Pivotado ao nível do Livro).


In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
print("Bibliotecas importadas com sucesso!")


Bibliotecas importadas com sucesso!


## 1. Carregamento do Dataset
Carregamos o dataset de livros limpo (`books_clean.parquet`) obtido nas etapas anteriores.


In [2]:
df = pd.read_parquet('../data/processed/books_clean.parquet')
print(f"Shape original: {df.shape}")
print(df[['title', 'author', 'genre_list']].head())


Shape original: (84054, 14)
                                               title  \
0  Between Two Fires: American Indians in the Civ...   
1                           Fashion Sourcebook 1920s   
2                                         Hungary 56   
3  All-American Anarchist: Joseph A. Labadie and ...   
4  The Human Equation: Building Profits by Puttin...   

                             author  \
0              Laurence M. Hauptman   
1  Charlotte Fiell,Emmanuelle Dirix   
2                     Andy Anderson   
3              Carlotta R. Anderson   
4                   Jeffrey Pfeffer   

                                          genre_list  
0  [history, military history, civil war, america...  
1    [couture, fashion, historical, art, nonfiction]  
2                                [politics, history]  
3                                   [labor, history]  
4  [business, leadership, romance, historical rom...  


## 2. Desnormalização (Explosão de Gêneros)
Utilizamos a função `.explode()` para desnormalizar o dataset, transformando listas de gêneros em linhas separadas. Cada livro passa a ter uma linha para cada um de seus gêneros listados.


In [3]:
# Explodir a coluna genre_list
df_exp = df.explode('genre_list').copy()
df_exp = df_exp.rename(columns={'genre_list': 'genre_single'})

# Remover registros sem gênero definido
df_exp = df_exp.dropna(subset=['genre_single'])

print(f"Shape após desnormalização (explosão): {df_exp.shape}")
print(f"Número de gêneros únicos no dataset: {df_exp['genre_single'].nunique()}")
print(df_exp[['title', 'genre_single']].head(10))


Shape após desnormalização (explosão): (790333, 14)
Número de gêneros únicos no dataset: 1179
                                               title          genre_single
0  Between Two Fires: American Indians in the Civ...               history
0  Between Two Fires: American Indians in the Civ...      military history
0  Between Two Fires: American Indians in the Civ...             civil war
0  Between Two Fires: American Indians in the Civ...      american history
0  Between Two Fires: American Indians in the Civ...    american civil war
0  Between Two Fires: American Indians in the Civ...            nonfiction
0  Between Two Fires: American Indians in the Civ...  north american hi...
0  Between Two Fires: American Indians in the Civ...      american history
0  Between Two Fires: American Indians in the Civ...      native americans
1                           Fashion Sourcebook 1920s               couture


### 2.1 Análise de Pareto dos Gêneros
A Análise de Pareto (regra 80/20) é aplicada aqui para justificar a necessidade de agrupar os gêneros. Mostraremos que uma pequena parcela dos gêneros concentra a maior parte das marcações do dataset.


In [4]:
# Calcular a distribuição acumulada de frequências
genre_counts = df_exp['genre_single'].value_counts().reset_index()
genre_counts.columns = ['genre', 'count']
genre_counts['cumulative_sum'] = genre_counts['count'].cumsum()
genre_counts['cumulative_pct'] = (genre_counts['cumulative_sum'] / genre_counts['count'].sum()) * 100

# Calcular quantos gêneros cobrem 80% do catálogo
pareto_80 = genre_counts[genre_counts['cumulative_pct'] <= 80]
num_genres_80 = len(pareto_80)
pct_genres_80 = (num_genres_80 / len(genre_counts)) * 100

print(f"Total de Gêneros Únicos: {len(genre_counts)}")
print(f"Número de Gêneros que cobrem 80% das marcações: {num_genres_80} ({pct_genres_80:.2f}%)")

# Plotar Gráfico de Pareto (Top 147 Gêneros que representam 80% das marcações)
top_genres = genre_counts.head(147)

fig_pareto = make_subplots(specs=[[{"secondary_y": True}]])

fig_pareto.add_trace(
    go.Bar(
        x=top_genres['genre'],
        y=top_genres['count'],
        name="Frequência",
        marker_color='plum'
    ),
    secondary_y=False
)

fig_pareto.add_trace(
    go.Scatter(
        x=top_genres['genre'],
        y=top_genres['cumulative_pct'],
        name="% Acumulado",
        mode='lines',
        marker_color='indigo'
    ),
    secondary_y=True
)

fig_pareto.update_layout(
    title_text="Análise de Pareto: Os 147 Gêneros que Concentram 80% das Marcações",
    template="plotly_white",
    height=600,
    showlegend=True
)

fig_pareto.update_xaxes(title_text="Gêneros (Passe o mouse sobre as barras para ver os nomes)", showticklabels=False)
fig_pareto.update_yaxes(title_text="Frequência (Barras)", secondary_y=False)
fig_pareto.update_yaxes(title_text="Percentual Acumulado (Linha)", secondary_y=True, range=[0, 105])

fig_pareto.show()


Total de Gêneros Únicos: 1179
Número de Gêneros que cobrem 80% das marcações: 150 (12.72%)


## 3. Agrupamento Hierárquico (Mapeamento para Gêneros Principais)
Para contornar a esparsidade dos mais de 1.100 gêneros sem precisar recorrer a técnicas cegas de redução de dimensionalidade (como SVD), criamos regras semânticas para agrupar subgêneros correlatos em **9 Gêneros Principais**.


In [5]:
def map_genre_to_main(genre):
    if not isinstance(genre, str):
        return 'Outros'
    g = genre.strip().lower()
    
    # 1. Romance
    if any(keyword in g for keyword in ['romance', 'love story', 'chick lit', 'erotica', 'shojo', 'menage', 'harlequin', 'regency', 'holiday', 'christmas', 'bdsm']):
        return 'Romance'
    
    # 2. Fantasia e Ficção Científica
    if any(keyword in g for keyword in ['fantasy', 'sci-fi', 'science fiction', 'magic', 'paranormal', 'supernatural', 'mythology', 'shapeshifters', 'dystopia', 'steampunk', 'space opera', 'aliens', 'fairies', 'witches', 'vampires', 'werewolves', 'urban fantasy', 'occult', 'apocalyptic', 'space', 'speculative fiction', 'folklore', 'myth', 'fables', 'fairy tale']):
        return 'Fantasia e Ficção Científica'
    
    # 3. Mistério, Thriller e Terror
    if any(keyword in g for keyword in ['mystery', 'thriller', 'horror', 'crime', 'suspense', 'detective', 'noir', 'spy', 'terror', 'slasher', 'murder', 'gothic', 'demons', 'ghosts', 'psychological thriller']):
        return 'Mistério, Thriller e Terror'
    
    # 4. História e Biografia
    if any(keyword in g for keyword in ['history', 'biography', 'autobiography', 'memoir', 'historical', 'military history', 'american history', 'world war', 'holocaust', 'genealogy', 'war', 'medieval', 'century', 'ancient', 'roman', 'royal', 'military']):
        return 'História e Biografia'
    
    # 5. Infantojuvenil e Quadrinhos
    if any(keyword in g for keyword in ['children', 'young adult', 'comics', 'graphic novel', 'sequential art', 'manga', 'middle grade', 'superhero', 'picture book', 'juvenile', 'kids', 'youth', 'fairy tales', 'school', 'teen', 'comic book', 'marvel', 'storytime', 'disney']):
        return 'Infantojuvenil e Quadrinhos'
    
    # 6. Não-Ficção e Autodesenvolvimento
    if any(keyword in g for keyword in ['nonfiction', 'non-fiction', 'philosophy', 'psychology', 'business', 'self help', 'science', 'academic', 'computer science', 'health', 'environment', 'politics', 'economics', 'sociology', 'religion', 'christianity', 'spirituality', 'theology', 'education', 'reference', 'writing', 'cookbooks', 'cultural', 'culture', 'anthropology', 'humanities', 'biology', 'mathematics', 'technology', 'textbooks', 'essays', 'feminism', 'christian', 'faith', 'church', 'buddhism', 'judaism', 'inspirational', 'leadership', 'programming', 'computers', 'finance', 'parenting', 'personal development', 'medicine', 'criticism', 'theory']):
        return 'Não-Ficção e Autodesenvolvimento'
    
    # 7. Artes, Lazer e Estilo de Vida
    if any(keyword in g for keyword in ['food', 'art', 'humor', 'sports', 'music', 'travel', 'games', 'crafts', 'gardening', 'design', 'photography', 'cooking', 'pets', 'animals', 'nature', 'comedy', 'diy', 'craft', 'hobbies']):
        return 'Artes, Lazer e Estilo de Vida'
    
    # 8. Ficção Geral e Literatura
    if any(keyword in g for keyword in ['fiction', 'literature', 'classics', 'contemporary', 'novel', 'short stories', 'antholog', 'drama', 'poetry', 'plays', 'adult', 'adventure', 'action', 'lgbt', 'gay', 'queer', 'glbt', 'yaoi', 'lesbian', 'transgender', 'gender', 'american', 'family', 'western', 'media tie in', 'language', 'womens', 'coming of age', 'literary']):
        return 'Ficção Geral e Literatura'
    
    return 'Outros'

# Aplicar o mapeamento de gêneros
df_exp['genre_main'] = df_exp['genre_single'].apply(map_genre_to_main)

# Exibir a distribuição pós-mapeamento
counts = df_exp['genre_main'].value_counts()
pcts = df_exp['genre_main'].value_counts(normalize=True) * 100

print("Distribuição dos Gêneros Principais Mapeados:")
for cat, val, pct in zip(counts.index, counts.values, pcts.values):
    print(f"  {cat:<35}: {val:>7} ({pct:.2f}%)")


Distribuição dos Gêneros Principais Mapeados:
  Não-Ficção e Autodesenvolvimento   :  154125 (19.50%)
  Ficção Geral e Literatura          :  140075 (17.72%)
  Outros                             :   88488 (11.20%)
  História e Biografia               :   83910 (10.62%)
  Fantasia e Ficção Científica       :   80827 (10.23%)
  Infantojuvenil e Quadrinhos        :   78620 (9.95%)
  Romance                            :   75670 (9.57%)
  Artes, Lazer e Estilo de Vida      :   49494 (6.26%)
  Mistério, Thriller e Terror        :   39124 (4.95%)


## 4. Análise Exploratória (EDA) pós-mapeamento
Vamos analisar a volumetria e as características de páginas e avaliações médias para cada gênero principal.


In [6]:
# Gráfico do Volume de Livros por Gênero Principal
fig_counts = px.bar(
    counts.reset_index(),
    x='count',
    y='genre_main',
    orientation='h',
    title='Volume de Livros por Gênero Principal (Dataset Explodido)',
    labels={'genre_main': 'Gênero Principal', 'count': 'Frequência de Livros'},
    color_discrete_sequence=['#2980B9'],
)
fig_counts.update_layout(yaxis={'categoryorder':'total ascending'}, height=500, template='plotly_white')
fig_counts.show()


## 4. Estruturas de Dados para a Clusterização (Abordagens A e B)

Para prosseguir com a modelagem do algoritmo K-Means, podemos estruturar os dados de duas maneiras distintas: a **Abordagem A (Explodida)** e a **Abordagem B (Pivotada)**.

### Abordagem A: Dataset Desnormalizado (Por Par Livro-Gênero)
Na **Abordagem A**, o dataset permanece no formato explodido (com 790.333 linhas). 
- **Estrutura:** Cada linha representa a relação individual de um livro com um gênero. Se um livro possui 3 gêneros principais, ele aparecerá em 3 linhas diferentes.
- **Utilidade:** É útil para análises granulares sobre gêneros específicos (ex: comparar diretamente Fantasia vs. Romance isoladamente).
- **Limitação:** Como o mesmo livro possui múltiplos registros duplicados no banco de dados, ele pode ser classificado em **diferentes clusters ao mesmo tempo**. Isso impede que tenhamos uma atribuição única e exclusiva de "Cluster" por livro.


In [7]:
print("Visualização da Abordagem A (Dataset Explodido):")
print(df_exp[['title', 'author', 'genre_main', 'rating', 'pages', 'totalratings']].head(10))


Visualização da Abordagem A (Dataset Explodido):
                                               title  \
0  Between Two Fires: American Indians in the Civ...   
0  Between Two Fires: American Indians in the Civ...   
0  Between Two Fires: American Indians in the Civ...   
0  Between Two Fires: American Indians in the Civ...   
0  Between Two Fires: American Indians in the Civ...   
0  Between Two Fires: American Indians in the Civ...   
0  Between Two Fires: American Indians in the Civ...   
0  Between Two Fires: American Indians in the Civ...   
0  Between Two Fires: American Indians in the Civ...   
1                           Fashion Sourcebook 1920s   

                             author                        genre_main  rating  \
0              Laurence M. Hauptman              História e Biografia    3.52   
0              Laurence M. Hauptman              História e Biografia    3.52   
0              Laurence M. Hauptman              História e Biografia    3.52   
0         

### Abordagem B: Dataset Pivotado (Ao Nível do Livro)
Na **Abordagem B**, nós re-agrupamos os dados de volta para o nível original de livro (retornando ao tamanho de 84.054 linhas).
- **Estrutura:** Fazemos o *pivot* (cruzamento) dos gêneros principais. Cada livro volta a ocupar **uma única linha**, e criamos **9 colunas binárias** (uma para cada gênero principal: `Romance`, `Fantasia e Ficção Científica`, etc.). Se o livro pertence àquele gênero, a coluna recebe o valor `1`, caso contrário, recebe `0`.
- **Utilidade:** Permite modelar o livro como uma entidade única. As 9 novas colunas binárias atuam como as "features de gênero" do livro, que serão combinadas com as variáveis numéricas (`rating`, `pages`, `totalratings`) para treinar o K-Means.

---

### Por que a Abordagem B é melhor para o nosso objetivo final?

Para o objetivo deste trabalho acadêmico (que é gerar uma coluna final que defina o qual único "Cluster Literário" cada livro pertence), a **Abordagem B é amplamente superior** à Abordagem A pelos seguintes motivos:

1. **Evita a Duplicidade e Conflito de Clusters:** Na Abordagem A, o livro *Drácula* poderia cair no Cluster 0 (pela linha de Terror) e no Cluster 2 (pela linha de Clássicos). Na Abordagem B, como o livro tem apenas **uma única linha**, o K-Means é forçado a colocá-lo em **um único cluster definitivo**, considerando todos os seus gêneros simultaneamente.
2. **Elimina a necessidade de SVD (Redução Matemática Cega):** Em vez de usar SVD para reduzir 1.179 colunas esparsas para 2 componentes abstratos, a Abordagem B reduz os gêneros a apenas 9 colunas extremamente limpas e legíveis.
3. **Interpretabilidade Acadêmica Máxima:** O perfilamento dos clusters fica muito mais fácil de explicar na sua apresentação de slides: podemos afirmar exatamente quais dos 9 gêneros principais definiram cada grupo.


In [8]:
# Criar colunas binárias para os 9 Gêneros Principais
df_pivot = pd.crosstab(
    df_exp['title'].reset_index(drop=True), 
    df_exp['genre_main'].reset_index(drop=True)
).clip(upper=1).reset_index()

# Agrupar colunas numéricas no nível do livro no dataset original
df_numeric_orig = df.groupby('title').agg(
    author=('author', 'first'),
    rating=('rating', 'mean'),
    pages=('pages', 'mean'),
    totalratings=('totalratings', 'mean')
).reset_index()

df_book_level = pd.merge(df_pivot, df_numeric_orig, on='title', how='inner')

print(f"Shape da Abordagem B (Nível do Livro): {df_book_level.shape}")
print(df_book_level.head())


Shape da Abordagem B (Nível do Livro): (81979, 14)
                                               title  \
0                                         "Daisuki."   
1                  "Dark Pictures" and Other Stories   
2             "Defects": Engendering the Modern Body   
3                              "Have-More" Plan, The   
4  "Headhunter" Hiring Secrets: The Rules of the ...   

   Artes, Lazer e Estilo de Vida  Fantasia e Ficção Científica  \
0                              0                             0   
1                              0                             0   
2                              0                             0   
3                              1                             0   
4                              0                             0   

   Ficção Geral e Literatura  História e Biografia  \
0                          1                     0   
1                          1                     0   
2                          0                     0   

## 5. Medição de Dispersão e Redução de Ruído (Entropia de Shannon)

Para avaliar matematicamente a dispersão dos dados e a concentração da informação, utilizamos a **Entropia de Shannon (H)**.

### O Diagnóstico do Trade-Off de Informação
A entropia quantifica a desordem ou incerteza nas categorias. 
- **Entropia Alta (7.73 bits - Original):** Representa um excesso de dispersão caótica e ruído (1.179 subgêneros esparsos).
- **Entropia Zero (0 bits):** Ocorrerria se agrupássemos tudo em apenas 1 gênero, eliminando a dispersão, mas também descartando *toda* a informação útil para diferenciação dos livros.
- **Entropia Moderada (3.05 bits - Nosso Mapeamento de 9 Gêneros):** Representa o ponto de equilíbrio (trade-off). Conseguimos reduzir a dispersão de ruído em **60.50%**, mas retivemos informação suficiente nas 9 categorias estruturadas de prateleiras literárias para viabilizar agrupamentos estatísticos úteis no K-Means.

Calculamos e comparamos os valores abaixo:


In [9]:
# Avaliar a redução de dispersão usando a Entropia de Shannon (H)
# H = -sum(p * log2(p))
# Uma entropia menor significa dados mais ordenados, concentrados e menos dispersos.
from scipy.stats import entropy

# Probabilidades de cada categoria
prob_original = df_exp['genre_single'].value_counts(normalize=True)
prob_consolidated = df_exp['genre_main'].value_counts(normalize=True)

# Calcular a Entropia de Shannon
entropy_orig = entropy(prob_original, base=2)
entropy_cons = entropy(prob_consolidated, base=2)

print("=== Avaliação Estatística de Dispersão ===")
print(f"Entropia de Shannon (Original - 1179 gêneros): {entropy_orig:.4f} bits")
print(f"Entropia de Shannon (Consolidado - 9 gêneros) : {entropy_cons:.4f} bits")
print(f"Redução na Entropia (Concentração da Informação): {((entropy_orig - entropy_cons) / entropy_orig) * 100:.2f}%")



=== Avaliação Estatística de Dispersão ===
Entropia de Shannon (Original - 1179 gêneros): 7.7371 bits
Entropia de Shannon (Consolidado - 9 gêneros) : 3.0562 bits
Redução na Entropia (Concentração da Informação): 60.50%


## 6. Exportação dos Dados Processados
Salvaremos os datasets das duas abordagens no diretório de dados processados para que possam ser utilizados nos próximos passos da modelagem de clusterização.


In [10]:
# Salvar no diretório de processados
df_exp.to_parquet('../data/processed/books_exploded_mapped.parquet', index=False)
df_book_level.to_parquet('../data/processed/books_pivot_mapped.parquet', index=False)
print("Arquivos salvos com sucesso na pasta data/processed!")


Arquivos salvos com sucesso na pasta data/processed!
